In [31]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openeo
import xarray as xr
import geopandas as gpd
import regionmask

In [2]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

interim_data_dir = project_root / "data" / "interim"
interim_data_dir.mkdir(parents=True, exist_ok=True)

In [8]:
connection = (
    openeo
    .connect("openeo.dataspace.copernicus.eu")
    .authenticate_oidc()
)

Authenticated using refresh token.


In [9]:
po_valley_extent = {
    "west": 7.5,
    "south": 44.3,
    "east": 13.0,
    "north": 46.7,
    "crs": "EPSG:4326",
}

In [10]:
def build_weekly_ch4_cube(year: int):
    """Build a weekly Sentinel-5P CH4 cube for one complete year."""
    
    temporal_extent = [
        f"{year}-01-01",
        f"{year + 1}-01-01",
    ]

    annual_cube = connection.load_collection(
        "SENTINEL_5P_L2",
        spatial_extent=po_valley_extent,
        temporal_extent=temporal_extent,
        bands=["CH4"],
    )

    weekly_cube = annual_cube.aggregate_temporal_period(
        period="week",
        reducer="mean",
    )

    return weekly_cube

In [11]:
ch4_weekly_2019 = build_weekly_ch4_cube(2019)

In [12]:
def get_annual_output_path(year: int) -> Path:
    return (
        interim_data_dir
        / f"sentinel5p_ch4_po_valley_weekly_{year}.nc"
    )

In [13]:
def download_ch4_year(year: int):
    """Execute and download one annual weekly CH4 batch job."""
    
    output_path = get_annual_output_path(year)

    if output_path.exists():
        print(f"{year}: file already available — skipping download.")
        return None

    weekly_cube = build_weekly_ch4_cube(year)

    job = weekly_cube.execute_batch(
        outputfile=output_path,
        out_format="NetCDF",
        title=f"Sentinel-5P weekly CH4 — Po Valley {year}",
    )

    print(f"{year}: saved to {output_path}")
    return job

In [14]:
job_2019 = download_ch4_year(2019)

2019: file already available — skipping download.


In [15]:
path_2019 = get_annual_output_path(2019)

ds_2019 = xr.open_dataset(path_2019)
ch4_2019 = ds_2019["CH4"]

ch4_2019

<xarray.DataArray 'CH4' (t: 53, y: 70, x: 101)> Size: 1MB
[374710 values with dtype=float32]
Coordinates:
  * t        (t) datetime64[ns] 424B 2018-12-30 2019-01-06 ... 2019-12-29
  * x        (x) float64 808B 7.527 7.582 7.636 7.691 ... 12.87 12.93 12.98
  * y        (y) float64 560B 46.68 46.65 46.61 46.58 ... 44.36 44.32 44.29
Attributes:
    long_name:     CH4
    units:         
    grid_mapping:  crs

In [11]:
print("Shape:", ch4_2019.shape)
print("Dimensions:", ch4_2019.dims)
print("First timestamp:", ch4_2019["t"].values[0])
print("Last timestamp:", ch4_2019["t"].values[-1])
print("Valid values:", int(ch4_2019.count().values))
print(
    "Missing proportion:",
    float(ch4_2019.isnull().mean().values),
)

Shape: (53, 70, 101)
Dimensions: ('t', 'y', 'x')
First timestamp: 2018-12-30T00:00:00.000000000
Last timestamp: 2019-12-29T00:00:00.000000000
Valid values: 97341
Missing proportion: 0.7402231058685383


In [16]:
path_2024 = (
    interim_data_dir
    / "sentinel5p_ch4_po_valley_weekly_2024.nc"
)

ds_2024 = xr.open_dataset(path_2024)

In [13]:
assert np.array_equal(
    ds_2019["x"].values,
    ds_2024["x"].values,
), "The 2019 and 2024 x coordinates differ."

assert np.array_equal(
    ds_2019["y"].values,
    ds_2024["y"].values,
), "The 2019 and 2024 y coordinates differ."

print("The 2019 and 2024 spatial grids match.")

The 2019 and 2024 spatial grids match.


In [14]:
print("2019 grid:", ds_2019.sizes["y"], "×", ds_2019.sizes["x"])
print("2024 grid:", ds_2024.sizes["y"], "×", ds_2024.sizes["x"])

2019 grid: 70 × 101
2024 grid: 70 × 101


In [17]:
for year in range(2020, 2024):
    download_ch4_year(year)

2020: file already available — skipping download.
2021: file already available — skipping download.
2022: file already available — skipping download.
2023: file already available — skipping download.


In [22]:
years = range(2019, 2025)

annual_datasets = {}
annual_diagnostics = []

reference_x = None
reference_y = None

for year in years:
    annual_path = (
        interim_data_dir
        / f"sentinel5p_ch4_po_valley_weekly_{year}.nc"
    )

    if not annual_path.exists():
        raise FileNotFoundError(
            f"Annual dataset not found: {annual_path}"
        )

    # I file sono abbastanza piccoli da poter essere caricati in memoria.
    ds = xr.load_dataset(annual_path)

    if "CH4" not in ds.data_vars:
        raise ValueError(
            f"CH4 not found in {annual_path.name}. "
            f"Available variables: {list(ds.data_vars)}"
        )

    ch4 = ds["CH4"]
    timestamps = pd.DatetimeIndex(ds["t"].values)

    time_difference_days = (
    pd.Series(timestamps)
    .diff()
    .dt.days
    .dropna()
)

    annual_diagnostics.append({
        "year": year,
        "timestamps": len(timestamps),
        "first_timestamp": timestamps.min(),
        "last_timestamp": timestamps.max(),
        "time_sorted": timestamps.is_monotonic_increasing,
        "timestamps_unique": timestamps.is_unique,
        "weekly_spacing": time_difference_days.eq(7).all(),
        "y_cells": ch4.sizes["y"],
        "x_cells": ch4.sizes["x"],
        "valid_values": int(ch4.count().values),
        "missing_proportion": float(
            ch4.isnull().mean().values
        ),
    })

    if reference_x is None:
        reference_x = ds["x"].values.copy()
        reference_y = ds["y"].values.copy()
    else:
        assert np.array_equal(
            ds["x"].values,
            reference_x,
        ), f"The x grid differs in {year}."

        assert np.array_equal(
            ds["y"].values,
            reference_y,
        ), f"The y grid differs in {year}."

    annual_datasets[year] = ds

annual_diagnostics_df = pd.DataFrame(
    annual_diagnostics
)

annual_diagnostics_df

,year,timestamps,first_timestamp,last_timestamp,time_sorted,timestamps_unique,weekly_spacing,y_cells,x_cells,valid_values,missing_proportion
0,2019,53,2018-12-30,2019-12-29,True,True,True,70,101,97341,0.740223
1,2020,53,2019-12-29,2020-12-27,True,True,True,70,101,93863,0.749505
2,2021,52,2021-01-03,2021-12-26,True,True,True,70,101,88225,0.760023
3,2022,50,2021-12-26,2022-12-25,True,True,False,70,101,93779,0.734713
4,2023,50,2023-01-01,2023-12-24,True,True,False,70,101,100073,0.716908
5,2024,53,2023-12-31,2024-12-29,True,True,True,70,101,92993,0.751827


In [23]:
for year in years:
    timestamps = pd.DatetimeIndex(
        annual_datasets[year]["t"].values
    )

    interval_df = pd.DataFrame({
        "previous_timestamp": timestamps[:-1],
        "next_timestamp": timestamps[1:],
    })

    interval_df["gap_days"] = (
        interval_df["next_timestamp"]
        - interval_df["previous_timestamp"]
    ).dt.days

    unexpected_gaps = interval_df.loc[
        interval_df["gap_days"] != 7
    ].copy()

    unexpected_gaps["missing_weekly_labels"] = (
        unexpected_gaps["gap_days"] // 7 - 1
    )

    print(f"\nYear {year}")
    
    if unexpected_gaps.empty:
        print("No internal temporal gaps.")
    else:
        print(unexpected_gaps.to_string(index=False))


Year 2019
No internal temporal gaps.

Year 2020
No internal temporal gaps.

Year 2021
No internal temporal gaps.

Year 2022
previous_timestamp next_timestamp  gap_days  missing_weekly_labels
        2022-07-24     2022-08-21        28                      3

Year 2023
previous_timestamp next_timestamp  gap_days  missing_weekly_labels
        2023-08-13     2023-08-27        14                      1
        2023-10-08     2023-10-22        14                      1

Year 2024
No internal temporal gaps.


In [24]:
all_timestamp_records = []

for year in years:
    timestamps = pd.DatetimeIndex(
        annual_datasets[year]["t"].values
    )

    annual_records = pd.DataFrame({
        "source_year": year,
        "timestamp": timestamps,
    })

    all_timestamp_records.append(annual_records)

all_timestamps_df = pd.concat(
    all_timestamp_records,
    ignore_index=True,
)

duplicate_timestamp_records = (
    all_timestamps_df.loc[
        all_timestamps_df["timestamp"].duplicated(
            keep=False
        )
    ]
    .sort_values(["timestamp", "source_year"])
)

duplicate_timestamp_records

,source_year,timestamp
52,2019,2019-12-29
53,2020,2019-12-29
157,2021,2021-12-26
158,2022,2021-12-26


In [ ]:
multiyear_output_path = (
    interim_data_dir
    / "sentinel5p_ch4_po_valley_weekly_2019_2024.nc"
)

if multiyear_output_path.exists():
    print(
        "Multiyear dataset already available — "
        "skipping download."
    )
    job_multiyear = None

else:
    multiyear_cube = connection.load_collection(
        "SENTINEL_5P_L2",
        spatial_extent=po_valley_extent,
        temporal_extent=[
            "2019-01-01",
            "2025-01-01",
        ],
        bands=["CH4"],
    )

    multiyear_weekly_cube = (
        multiyear_cube
        .aggregate_temporal_period(
            period="week",
            reducer="mean",
        )
    )

    job_multiyear = multiyear_weekly_cube.execute_batch(
        outputfile=multiyear_output_path,
        out_format="NetCDF",
        title=(
            "Sentinel-5P weekly CH4 — "
            "Po Valley 2019–2024"
        ),
    )

    print("Saved to:", multiyear_output_path)

0:00:00 Job 'j-2607241253374eb6afa931e0f9e66f96': send 'start'
0:00:03 Job 'j-2607241253374eb6afa931e0f9e66f96': queued (progress 0%)
0:00:08 Job 'j-2607241253374eb6afa931e0f9e66f96': queued (progress 0%)
0:00:15 Job 'j-2607241253374eb6afa931e0f9e66f96': queued (progress 0%)
0:00:23 Job 'j-2607241253374eb6afa931e0f9e66f96': queued (progress 0%)
0:00:32 Job 'j-2607241253374eb6afa931e0f9e66f96': queued (progress 0%)
0:00:45 Job 'j-2607241253374eb6afa931e0f9e66f96': queued (progress 0%)
0:01:00 Job 'j-2607241253374eb6afa931e0f9e66f96': running (progress N/A)
0:01:19 Job 'j-2607241253374eb6afa931e0f9e66f96': running (progress N/A)
0:01:43 Job 'j-2607241253374eb6afa931e0f9e66f96': running (progress N/A)
0:02:14 Job 'j-2607241253374eb6afa931e0f9e66f96': running (progress N/A)
0:02:51 Job 'j-2607241253374eb6afa931e0f9e66f96': running (progress N/A)
0:03:38 Job 'j-2607241253374eb6afa931e0f9e66f96': running (progress N/A)
0:04:36 Job 'j-2607241253374eb6afa931e0f9e66f96': running (progress N/A)


In [26]:
multiyear_ds = xr.load_dataset(multiyear_output_path)

multiyear_times = pd.DatetimeIndex(
    multiyear_ds["t"].values
).sort_values()

duplicate_times = multiyear_times[
    multiyear_times.duplicated(keep=False)
]

complete_weekly_index = pd.date_range(
    start=multiyear_times.min(),
    end=multiyear_times.max(),
    freq="7D",
)

missing_times = complete_weekly_index.difference(
    multiyear_times
)

print("Total timestamps:", len(multiyear_times))
print("Unique timestamps:", multiyear_times.nunique())
print("First timestamp:", multiyear_times.min())
print("Last timestamp:", multiyear_times.max())

print("\nDuplicate timestamps:")
print(duplicate_times)

print("\nMissing timestamps:")
print(missing_times)

Total timestamps: 309
Unique timestamps: 309
First timestamp: 2018-12-30 00:00:00
Last timestamp: 2024-12-29 00:00:00

Duplicate timestamps:
DatetimeIndex([], dtype='datetime64[ns]', freq=None)

Missing timestamps:
DatetimeIndex(['2022-07-31', '2022-08-07', '2022-08-14', '2023-08-20',
               '2023-10-15'],
              dtype='datetime64[ns]', freq=None)


In [27]:
complete_weekly_index = pd.date_range(
    start=multiyear_times.min(),
    end=multiyear_times.max(),
    freq="7D",
)

ch4_multiyear = (
    multiyear_ds["CH4"]
    .sortby("t")
    .reindex(t=complete_weekly_index)
)

In [29]:
complete_times = pd.DatetimeIndex(
    ch4_multiyear["t"].values
)

time_difference_days = (
    pd.Series(complete_times)
    .diff()
    .dt.days
    .dropna()
)

print("Total timestamps:", len(complete_times))
print("Unique timestamps:", complete_times.nunique())
print("Weekly spacing:", time_difference_days.eq(7).all())
print("Grid shape:", ch4_multiyear.sizes["y"],
      "×", ch4_multiyear.sizes["x"])

assert len(complete_times) == 314
assert complete_times.is_unique
assert time_difference_days.eq(7).all()
assert ch4_multiyear.sizes["y"] == 70
assert ch4_multiyear.sizes["x"] == 101

Total timestamps: 314
Unique timestamps: 314
Weekly spacing: True
Grid shape: 70 × 101


In [30]:
inserted_weeks = pd.DatetimeIndex([
    "2022-07-31",
    "2022-08-07",
    "2022-08-14",
    "2023-08-20",
    "2023-10-15",
])

for week in inserted_weeks:
    week_is_all_nan = bool(
        ch4_multiyear
        .sel(t=week)
        .isnull()
        .all()
        .values
    )

    print(week.date(), "all NaN:", week_is_all_nan)
    assert week_is_all_nan

2022-07-31 all NaN: True
2022-08-07 all NaN: True
2022-08-14 all NaN: True
2023-08-20 all NaN: True
2023-10-15 all NaN: True


In [35]:
roi_path = (
    project_root
    / "data"
    / "regions"
    / "po_valley_provisional_envelope_roi.geojson"
)

roi_gdf = gpd.read_file(roi_path)

print("Original ROI CRS:", roi_gdf.crs)

roi_gdf = roi_gdf.to_crs("EPSG:4326")

Original ROI CRS: EPSG:4326


In [36]:
roi_mask = regionmask.mask_geopandas(
    roi_gdf,
    ch4_multiyear["x"],
    ch4_multiyear["y"],
)

inside_roi = roi_mask.notnull()

number_of_roi_cells = int(
    inside_roi.sum().values
)

print("Number of cells inside ROI:", number_of_roi_cells)

Number of cells inside ROI: 2974


In [37]:
ch4_multiyear_roi = (
    ch4_multiyear
    .where(inside_roi)
)

In [38]:
coverage_training_data = ch4_multiyear_roi.sel(
    t=slice("2018-12-30", "2021-12-26")
)

In [39]:
training_temporal_coverage = (
    coverage_training_data
    .notnull()
    .mean(dim="t")
)

In [40]:
minimum_temporal_coverage = 0.25

candidate_mask = (
    inside_roi
    & (
        training_temporal_coverage
        >= minimum_temporal_coverage
    )
)

number_of_candidate_cells = int(
    candidate_mask.sum().values
)

print("ROI cells:", number_of_roi_cells)
print("Candidate cells:", number_of_candidate_cells)

ROI cells: 2974
Candidate cells: 2620


In [41]:
ch4_multiyear_candidate = (
    ch4_multiyear
    .where(candidate_mask)
)

In [42]:
multiyear_candidate_ds = xr.Dataset(
    {
        "CH4": ch4_multiyear_candidate,
        "roi_mask": inside_roi.astype("int8"),
        "candidate_mask": candidate_mask.astype("int8"),
        "training_temporal_coverage": (
            training_temporal_coverage
        ),
    }
)

In [43]:
multiyear_candidate_ds.attrs.update({
    "description": (
        "Weekly Sentinel-5P CH4 observations over "
        "candidate cells in the Po Valley, 2019–2024."
    ),
    "source": (
        "Copernicus Data Space Ecosystem openEO, "
        "SENTINEL_5P_L2, CH4 band."
    ),
    "temporal_extent": "2019-01-01 to 2025-01-01",
    "temporal_frequency": "Weekly",
    "roi_file": "po_valley_provisional_roi.geojson",
    "coverage_reference_period": "2019–2021",
    "minimum_temporal_coverage": 0.25,
    "number_of_roi_cells": number_of_roi_cells,
    "number_of_candidate_cells": number_of_candidate_cells,
    "inserted_missing_weeks": (
        "2022-07-31, 2022-08-07, 2022-08-14, "
        "2023-08-20, 2023-10-15"
    ),
})

In [44]:
multiyear_candidate_ds["roi_mask"].attrs.update({
    "description": (
        "Spatial mask identifying cells inside "
        "the provisional Po Valley ROI."
    ),
    "values": "1 = inside ROI, 0 = outside ROI",
})

multiyear_candidate_ds["candidate_mask"].attrs.update({
    "description": (
        "Cells inside the ROI with temporal coverage "
        "of at least 25% during 2019–2021."
    ),
    "values": "1 = candidate cell, 0 = excluded cell",
})

multiyear_candidate_ds[
    "training_temporal_coverage"
].attrs.update({
    "description": (
        "Proportion of non-missing weekly observations "
        "during the 2019–2021 coverage reference period."
    ),
    "range": "0 to 1",
})

In [45]:
multiyear_candidate_path = (
    interim_data_dir
    / "sentinel5p_ch4_po_valley_candidate_2019_2024.nc"
)

multiyear_candidate_ds.to_netcdf(
    multiyear_candidate_path,
    mode="w",
)

print("Saved to:", multiyear_candidate_path)

Saved to: c:\Users\Pietro\OneDrive\Desktop\Po-Valley-Methane-Forecasting\data\interim\sentinel5p_ch4_po_valley_candidate_2019_2024.nc


In [46]:
saved_candidate_ds = xr.open_dataset(
    multiyear_candidate_path
)

print(saved_candidate_ds)

assert saved_candidate_ds.sizes["t"] == 314

assert int(
    saved_candidate_ds["roi_mask"].sum().values
) == 2974

assert int(
    saved_candidate_ds["candidate_mask"].sum().values
) == 2620

print("Saved candidate dataset successfully verified.")

saved_candidate_ds.close()

<xarray.Dataset> Size: 9MB
Dimensions:                     (t: 314, x: 101, y: 70)
Coordinates:
  * t                           (t) datetime64[ns] 3kB 2018-12-30 ... 2024-12-29
  * x                           (x) float64 808B 7.527 7.582 ... 12.93 12.98
  * y                           (y) float64 560B 46.68 46.65 ... 44.32 44.29
Data variables:
    CH4                         (t, y, x) float32 9MB ...
    roi_mask                    (y, x) int8 7kB ...
    candidate_mask              (y, x) int8 7kB ...
    training_temporal_coverage  (y, x) float64 57kB ...
Attributes:
    description:                Weekly Sentinel-5P CH4 observations over cand...
    source:                     Copernicus Data Space Ecosystem openEO, SENTI...
    temporal_extent:            2019-01-01 to 2025-01-01
    temporal_frequency:         Weekly
    roi_file:                   po_valley_provisional_roi.geojson
    coverage_reference_period:  2019–2021
    minimum_temporal_coverage:  0.25
    number_of_roi_ce